<a href="https://colab.research.google.com/github/Madhav-Sharma91/RFL-python-internship/blob/main/day30project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install pdfplumber
import re
import io
from datetime import datetime

import pandas as pd
import pdfplumber


# ============================================================
# DATE HELPERS
# ============================================================

DATE_PATTERNS = [
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
    r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b",
    r"\b\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4}\b",
]


def extract_date(text, keyword=None):
    """
    Extract a date from text.
    If keyword is provided, search near that keyword first.
    """

    search_text = text

    if keyword:
        pattern = rf"{re.escape(keyword)}\s*[:\-]?\s*(.{{0,50}})"
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            search_text = match.group(1)

    for pattern in DATE_PATTERNS:

        match = re.search(
            pattern,
            search_text,
            flags=re.IGNORECASE
        )

        if match:
            value = match.group(0)

            parsed = pd.to_datetime(
                value,
                errors="coerce",
                dayfirst=True
            )

            if not pd.isna(parsed):
                return parsed

    return pd.NaT


# ============================================================
# TEXT EXTRACTION
# ============================================================

def extract_pdf_text(file_bytes):
    """
    Extract text from a text-based PDF.
    """

    text = ""

    with pdfplumber.open(
        io.BytesIO(file_bytes)
    ) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += "\n" + page_text

    return text


# ============================================================
# INVOICE NUMBER
# ============================================================

def extract_invoice_number(text):

    patterns = [
        r"invoice\s*(?:number|no\.?|#)\s*[:\-]?\s*([A-Za-z0-9\-/]+)",
        r"inv\s*(?:number|no\.?|#)\s*[:\-]?\s*([A-Za-z0-9\-/]+)",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1).strip()

    return "Unknown"


# ============================================================
# CUSTOMER DETAILS
# ============================================================

def extract_customer_details(text):

    patterns = [
        r"(?:customer|client|bill\s*to|billed\s*to)\s*[:\-]?\s*(.+)",
        r"(?:customer\s*name|client\s*name)\s*[:\-]?\s*(.+)",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            customer = match.group(1).strip()

            # Stop at common field labels.
            customer = re.split(
                r"\s+(?:date|invoice|address|email|phone|total)\s*[:\-]",
                customer,
                flags=re.IGNORECASE
            )[0]

            return customer.strip()

    return "Unknown"


# ============================================================
# CUSTOMER EMAIL
# ============================================================

def extract_email(text):

    match = re.search(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        text
    )

    if match:
        return match.group(0)

    return ""


# ============================================================
# CUSTOMER PHONE
# ============================================================

def extract_phone(text):

    match = re.search(
        r"(?:\+?\d[\d\s\-()]{8,}\d)",
        text
    )

    if match:
        return match.group(0).strip()

    return ""


# ============================================================
# TOTAL AMOUNT
# ============================================================

def extract_total(text):

    patterns = [
        r"(?:grand\s+total|total\s+amount|amount\s+due|total)"
        r"\s*[:\-]?\s*(?:₹|Rs\.?|INR|\$)?\s*([\d,]+(?:\.\d{1,2})?)"
    ]

    matches = re.findall(
        patterns[0],
        text,
        flags=re.IGNORECASE
    )

    if matches:

        values = []

        for value in matches:

            try:
                values.append(
                    float(value.replace(",", ""))
                )
            except ValueError:
                pass

        if values:
            return max(values)

    return 0.0


# ============================================================
# ITEM EXTRACTION
# ============================================================

def extract_items(text):

    items = []

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    for line in lines:

        # Example:
        # Laptop 2 50000 100000
        #
        # Product | Qty | Price | Total

        pattern = (
            r"^(.+?)\s+"
            r"(\d+(?:\.\d+)?)\s+"
            r"(?:₹|Rs\.?|INR|\$)?\s*"
            r"([\d,]+(?:\.\d{1,2})?)\s+"
            r"(?:₹|Rs\.?|INR|\$)?\s*"
            r"([\d,]+(?:\.\d{1,2})?)$"
        )

        match = re.match(
            pattern,
            line,
            flags=re.IGNORECASE
        )

        if match:

            description = match.group(1).strip()

            quantity = float(
                match.group(2)
            )

            unit_price = float(
                match.group(3).replace(",", "")
            )

            total = float(
                match.group(4).replace(",", "")
            )

            items.append({
                "Description": description,
                "Quantity": quantity,
                "Unit_Price": unit_price,
                "Item_Total": total
            })

    return items


# ============================================================
# SINGLE PDF PROCESSOR
# ============================================================

def process_pdf(file_bytes, filename="invoice.pdf"):

    text = extract_pdf_text(file_bytes)

    invoice_number = extract_invoice_number(
        text
    )

    customer = extract_customer_details(
        text
    )

    email = extract_email(text)

    phone = extract_phone(text)

    invoice_date = extract_date(
        text,
        "invoice date"
    )

    if pd.isna(invoice_date):

        invoice_date = extract_date(
            text,
            "date"
        )

    due_date = extract_date(
        text,
        "due date"
    )

    total = extract_total(
        text
    )

    items = extract_items(
        text
    )

    calculated_total = sum(
        item["Item_Total"]
        for item in items
    )

    # If item totals were successfully extracted,
    # prefer the calculated total.
    if calculated_total > 0:
        final_total = calculated_total
    else:
        final_total = total

    return {
        "Source_File": filename,
        "Invoice_Number": invoice_number,
        "Customer": customer,
        "Email": email,
        "Phone": phone,
        "Invoice_Date": invoice_date,
        "Due_Date": due_date,
        "Total_Amount": final_total,
        "Items": items,
        "Raw_Text": text
    }


# ============================================================
# CSV PROCESSOR
# ============================================================

def process_csv(df):

    df = df.copy()

    # Normalize common column names.

    rename_map = {}

    for column in df.columns:

        normalized = (
            str(column)
            .strip()
            .lower()
            .replace(" ", "_")
        )

        if normalized in [
            "invoice_no",
            "invoice_number",
            "invoice_id"
        ]:
            rename_map[column] = "Invoice_Number"

        elif normalized in [
            "customer",
            "customer_name",
            "client",
            "client_name"
        ]:
            rename_map[column] = "Customer"

        elif normalized in [
            "invoice_date",
            "date"
        ]:
            rename_map[column] = "Invoice_Date"

        elif normalized in [
            "due_date",
            "payment_due_date"
        ]:
            rename_map[column] = "Due_Date"

        elif normalized in [
            "total",
            "total_amount",
            "amount"
        ]:
            rename_map[column] = "Total_Amount"

    df = df.rename(
        columns=rename_map
    )

    required = [
        "Invoice_Number",
        "Customer",
        "Invoice_Date",
        "Due_Date",
        "Total_Amount"
    ]

    for column in required:

        if column not in df.columns:
            df[column] = ""

    df["Invoice_Date"] = pd.to_datetime(
        df["Invoice_Date"],
        errors="coerce"
    )

    df["Due_Date"] = pd.to_datetime(
        df["Due_Date"],
        errors="coerce"
    )

    df["Total_Amount"] = pd.to_numeric(
        df["Total_Amount"],
        errors="coerce"
    ).fillna(0)

    df["Source_File"] = "CSV"

    return df


# ============================================================
# OVERDUE DETECTION
# ============================================================

def identify_overdue(
    df,
    as_of_date=None
):

    result = df.copy()

    if as_of_date is None:
        as_of_date = pd.Timestamp.today().normalize()

    result["Days_Overdue"] = (
        as_of_date - result["Due_Date"]
    ).dt.days

    result["Overdue"] = (
        result["Due_Date"].notna()
        & (result["Days_Overdue"] > 0)
    )

    result["Days_Overdue"] = (
        result["Days_Overdue"]
        .fillna(0)
        .clip(lower=0)
        .astype(int)
    )

    return result


# ============================================================
# CONSOLIDATED REPORT
# ============================================================

def create_report(
    invoices,
    as_of_date=None
):

    report = invoices.copy()

    report = identify_overdue(
        report,
        as_of_date
    )

    report["Status"] = np_where_status(
        report
    )

    return report


def np_where_status(df):

    status = []

    for _, row in df.iterrows():

        if row["Overdue"]:
            status.append("Overdue")

        else:
            status.append("Current")

    return status

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 139.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 149.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 124.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
